In [20]:
# 2 by 2 grid world, initial state values
S = [0, 0, 0, 0]
policy = [0, 0, 0, 0]
q_values = [[0, 0, 0, 0] for _ in range(4)]
k = 5

def state_update(state, action):
    # state transition function
    if action == 0:  # up
        if state in [2, 3]:
            return state - 2
        return state
    if action == 1:  # down
        if state in [0, 1]:
            return state + 2
        return state
    if action == 2:  # left
        if state in [1, 3]:
            return state - 1    
        else:
            return state
    if action == 3:  # right
        if state in [0, 2]:
            return state + 1
        else:
            return state

# Policy iteration algorithm
delta_v = 100
initial_policy = [1, 1, 1, 1]
q_values = [[0, 0, 0, 0] for _ in range(4)]

# initialize state values
S = [0, 0, 0, 0]

i = 0
convergence = []

# policy evaluation
while(i < 10):
    delta_v = 0
    j = 0
    while(j < 10):
        for s in range(4):
            initial_value = S[s]
            # update state value
            S[s] = q_values[s][initial_policy[s]]
            
            # update q-values based on the updated state values
            a = initial_policy[s]
            next_state = state_update(s, a)
            reward = 1 if next_state == 3 else 0
            q_values[s][a] = reward + 0.9 * S[next_state]
            
            delta_v = max(delta_v, abs(initial_value - S[s]))
        j += 1
        convergence.append(delta_v)
        if delta_v < 0.8:
            break
       

    for s in range(4):
        for a in range(4):
            next_state = state_update(s, a)
            reward = 1 if next_state == 3 else 0
            q_values[s][a] = reward + 0.9 * S[next_state]
            # update policy based on the updated q-values
            policy[s] = q_values[s].index(max(q_values[s]))
    
    i += 1 
    print(f"Iteration {i}: State {s}, Value: {S[s]:.2f}, Q-values: {q_values[s]}", end="\n")
    
# Print the optimal policy
print("Optimal Policy:")
for s in range(4):
    print(f"State {s}: Action {policy[s]}")

# Print the state values
print("\nState Values:")
for s in range(4):
    print(f"State {s}: Value {S[s]}")
    
# Print the Q-values
print("\nQ-values:")
for s in range(4):
    print(f"State {s}: {q_values[s]}")

Iteration 1: State 3, Value: 0.00, Q-values: [0.0, 1.0, 0.0, 1.0]
Iteration 2: State 3, Value: 6.51, Q-values: [5.5132155990000005, 6.861894039100001, 0.0, 6.861894039100001]
Iteration 3: State 3, Value: 6.86, Q-values: [6.175704635190001, 7.175704635190001, 0.0, 7.175704635190001]
Iteration 4: State 3, Value: 7.18, Q-values: [6.458134171671, 7.458134171671, 0.0, 7.458134171671]
Iteration 5: State 3, Value: 7.46, Q-values: [6.7123207545039, 7.7123207545039, 0.0, 7.7123207545039]
Iteration 6: State 3, Value: 7.71, Q-values: [6.94108867905351, 7.94108867905351, 0.0, 7.94108867905351]
Iteration 7: State 3, Value: 7.94, Q-values: [7.146979811148159, 8.14697981114816, 0.0, 8.14697981114816]
Iteration 8: State 3, Value: 8.15, Q-values: [7.332281830033344, 8.332281830033345, 0.0, 8.332281830033345]
Iteration 9: State 3, Value: 8.33, Q-values: [7.499053647030011, 8.49905364703001, 0.0, 8.49905364703001]
Iteration 10: State 3, Value: 8.50, Q-values: [7.649148282327009, 8.649148282327008, 0.0, 8

In [8]:
class GridWorldPolicyIteration:
    def __init__(self, width, height, rewards=None, gamma=0.9, theta=1e-3):
        self.width = width
        self.height = height
        self.n_tiles = width * height
        self.rewards = list(rewards) if rewards is not None else [0.0] * self.n_tiles
        if len(self.rewards) != self.n_tiles:
            raise ValueError("rewards must have length width * height")
        self.gamma = gamma
        self.theta = theta
        self.n_actions = 4
        self.action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}
        self.q_values = [[0.0] * self.n_actions for _ in range(self.n_tiles)]
        self.policy = [0] * self.n_tiles
        self.values = [0.0] * self.n_tiles

    def to_index(self, row, col):
        return row * self.width + col

    def to_coord(self, state):
        return divmod(state, self.width)

    def next_state(self, state, action):
        row, col = self.to_coord(state)
        if action == 0:
            row = max(0, row - 1)
        elif action == 1:
            row = min(self.height - 1, row + 1)
        elif action == 2:
            col = max(0, col - 1)
        elif action == 3:
            col = min(self.width - 1, col + 1)
        return self.to_index(row, col)

    def policy_evaluation(self, max_iterations=1000):
        for _ in range(max_iterations):
            delta = 0.0
            for state in range(self.n_tiles):
                action = self.policy[state]
                ns = self.next_state(state, action)
                reward = self.rewards[ns]
                new_value = reward + self.gamma * self.values[ns]
                delta = max(delta, abs(self.values[state] - new_value))
                self.values[state] = new_value
            if delta < self.theta:
                break

    def policy_improvement(self):
        policy_stable = True
        for state in range(self.n_tiles):
            old_action = self.policy[state]
            action_values = []
            for action in range(self.n_actions):
                ns = self.next_state(state, action)
                reward = self.rewards[ns]
                action_values.append(reward + self.gamma * self.values[ns])
            best_action = action_values.index(max(action_values))
            self.policy[state] = best_action
            if best_action != old_action:
                policy_stable = False
            self.q_values[state] = action_values
        return policy_stable

    def run(self, max_iterations=20):
        for i in range(max_iterations):
            self.policy_evaluation()
            if self.policy_improvement():
                break
        return i + 1

    def policy_grid(self):
        return [[self.action_symbols[self.policy[self.to_index(row, col)]]
                 for col in range(self.width)]
                for row in range(self.height)]

    def values_grid(self):
        return [[round(self.values[self.to_index(row, col)], 2)
                 for col in range(self.width)]
                for row in range(self.height)]

    def print_policy(self):
        print("Learned policy:")
        for row in self.policy_grid():
            print(" ".join(row))
        print("\nState values:")
        for row in self.values_grid():
            print(" ".join(f"{v:.2f}" for v in row))


# Example usage:
rewards = [0.0, -1.0, 0.0, 0.0,
           0.0, 0.0, 0.0, -1.0,
           0.0, -1.0, 0.0, -1.0,
           0.0, 0.0, 0.0, 1.0]
agent = GridWorldPolicyIteration(width=4, height=4, rewards=rewards, gamma=0.9)
iterations = agent.run(max_iterations=20)
print(f"Policy iteration finished in {iterations} iterations")
agent.print_policy()

Policy iteration finished in 8 iterations
Learned policy:
↓ ↓ ↓ ←
↓ → ↓ ↓
↓ ↓ ↓ ↓
→ → → ↓

State values:
5.90 6.56 7.29 6.56
6.56 7.29 8.10 8.00
7.29 8.10 9.00 10.00
8.10 9.00 10.00 10.00
